# Fixed-Dense-SW versus matched shuffled control — seed 3, T4×2
Inputs: one completed seed-3 U/R/SW/RG output (`e2e_pairwise_pilot_v2`), the small output from `kaggle_rq2_fixed_dense_sw_tau_probe_cpu.ipynb`, CIFAR-100 Python-format dataset, and Kaggle secret `github_token`. Uniform E10→E100 is reused as the existing anchor; only the two new branches train. Both use the same common E10 model/optimizer/scheduler/RNG. Set `SELECTED_KAPPA` only after inspecting the CPU probe; it is frozen before training.

In [ ]:
import os,subprocess,sys,tempfile,importlib,shutil,zipfile,json,time
from pathlib import Path
from IPython.display import display
from kaggle_secrets import UserSecretsClient
PROJECT_ROOT=Path('/kaggle/working/new-pruning')
token=UserSecretsClient().get_secret('github_token');assert token,'Missing github_token secret'
with tempfile.TemporaryDirectory() as temporary:
    askpass=Path(temporary)/'askpass.py'
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env=os.environ.copy();env.update(GIT_ASKPASS=str(askpass),GIT_TERMINAL_PROMPT='0',GITHUB_TOKEN_RUNTIME=token)
    subprocess.run(['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)],env=env,check=True)
token=None;os.chdir(PROJECT_ROOT);sys.path.insert(0,str(PROJECT_ROOT))
import torch,pandas as pd
assert torch.cuda.device_count()==2,f'Select T4 x2; found {torch.cuda.device_count()}'
import rq2_e2e_pairwise_pilot as pilot
import rq2_fixed_dense_sw_gate as gate
import scripts.run_fixed_dense_sw_gate as runner
assert hasattr(gate,'freeze_policy') and hasattr(runner,'run_workers'),'Pull the new fixed-dense-SW source revision'
GIT_COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip();print('Code:',GIT_COMMIT)


In [ ]:
INPUT_ROOT=Path('/kaggle/input')
DATASET_ROOT=pilot.find_cifar100_root(INPUT_ROOT)
ROOT=pilot.materialize_progress(INPUT_ROOT,'/kaggle/working/e2e_pairwise_pilot_v2','/kaggle/working/materialized-fixedsw-seed3')
required=[ROOT/'frozen_protocol.json',ROOT/'resolved_config.yaml',ROOT/'common_warmup/epoch_010.pt',ROOT/'pure_sw/sw_policies/epoch_010.npz',ROOT/'pure_sw/training_provenance.json',ROOT/'uniform/checkpoints/epoch_100.pt',ROOT/'uniform/dense_metrics.csv',ROOT/'uniform/training_provenance.json']
missing=[str(p) for p in required if not p.is_file()];assert not missing,missing
probe_sources=sorted(INPUT_ROOT.rglob('fixed_dense_sw_tau_source.json'))
assert len(probe_sources)==1,f'Expected one E10 CPU probe, found {probe_sources}'
PROBE_DIR=probe_sources[0].parent
probe=pd.read_csv(PROBE_DIR/'fixed_dense_sw_tau_probe.csv')
display(probe[['kappa','tau','entropy','effective_support','support_size','l1_to_uniform','max_probability']])
print('ROOT:',ROOT,'CIFAR:',DATASET_ROOT,'PROBE:',PROBE_DIR)


## Freeze one τ before E2E
Set κ to **one of 0.1, 1.0, 10.0** based solely on the displayed offline policy shape; do not inspect E2E accuracy to change it. The shuffled policy is a width-label permutation of the selected q, so its entropy, support and marginals are exactly matched.

In [ ]:
SELECTED_KAPPA = None  # Choose 0.1, 1.0, or 10.0 after reviewing the CPU probe.
assert SELECTED_KAPPA in gate.KAPPAS,'Freeze κ from the offline E10 probe before training'
protocol=gate.freeze_policy(ROOT,PROBE_DIR,SELECTED_KAPPA)
shutil.copy2(PROBE_DIR/'fixed_dense_sw_tau_probe.csv',ROOT/'fixed_dense_sw_tau_probe.csv')
shutil.copy2(PROBE_DIR/'fixed_dense_sw_tau_source.json',ROOT/'fixed_dense_sw_tau_source.json')
print('Frozen κ:',protocol['selected_kappa'],'τ:',protocol['tau'])
display(pd.DataFrame(protocol['policy_summaries']).T)
print('Matched-control diagnostics:',protocol['control_diagnostics'])


In [ ]:
started=time.perf_counter()
runtime=runner.run_workers(ROOT,DATASET_ROOT,gpu_ids=(0,1))
display(runtime)
comparison=gate.summarize_gate(ROOT)
display(comparison)
print('Seed-3 development gate; validation only. Wall hours:',round((time.perf_counter()-started)/3600,2))


In [ ]:
# Small analysis bundle: no checkpoints. Full resumable state stays in the Kaggle output tree.
files=[ROOT/'fixed_dense_sw_frozen_protocol.json',ROOT/'fixed_dense_sw_tau_probe.csv',ROOT/'fixed_dense_sw_policy_summary.csv',ROOT/'fixed_dense_sw_gate_comparison.csv',ROOT/'fixed_dense_sw_gate_summary.json',ROOT/'fixed_dense_sw_runtime.csv']
for method in ('fixed_dense_sw','fixed_dense_shuffled'):
    files += [ROOT/method/name for name in ('dense_metrics.csv','pair_stats.csv','train_log.csv','policy_summary.json','width_marginals.csv','training_provenance.json')]
files += [ROOT/'uniform/dense_metrics.csv',ROOT/'uniform/pair_stats.csv',ROOT/'uniform/metrics.csv']
assert all(p.is_file() for p in files),[str(p) for p in files if not p.is_file()]
bundle=Path('/kaggle/working/rq2-fixed-dense-sw-seed3-analysis.zip')
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files:archive.write(path,path.relative_to(ROOT))
print('Analysis-only ZIP:',bundle,'bytes:',bundle.stat().st_size)
